# TP 2 — Qualité des données et jointures (santé)
**Durée indicative : 55–70 min**

Objectifs : lire des données synthétiques, repérer des anomalies, filtrer, dédupliquer et joindre deux tables.

> **Environment:** Databricks notebook in a Git folder. Attach a compute resource with PySpark available. Data files are in the adjacent `data/` folder. This lab uses only **synthetic** data.

> **Expected skill level:** first experience with Big Data / Spark. Follow the cells in order.


In [ ]:
from pathlib import Path
import csv
from pyspark.sql import functions as F

# Databricks Runtime 14+ sets the current directory to the notebook's folder.
DATA_DIR = Path.cwd().parent / "data"
assert DATA_DIR.is_dir(), f"Data directory not found: {DATA_DIR}. Open the notebook from the Git folder."

def read_small_csv(filename):
    # Python reads the small Git-tracked CSV on the driver; Spark does the processing.
    # For big data in production, read directly from cloud storage instead.
    with (DATA_DIR / filename).open(encoding="utf-8", newline="") as handle:
        rows = list(csv.DictReader(handle))
    return spark.createDataFrame(rows)


In [ ]:
admissions_raw = read_small_csv("admissions_synthetiques.csv")
patients_raw = read_small_csv("patients_synthetiques.csv")
admissions_raw.show(5)
patients_raw.show(5)
admissions_raw.printSchema()

## 1. Construire des types cohérents
Les CSV contiennent du texte. Transformer les colonnes utiles en `date` et `int`.

In [ ]:
admissions = (admissions_raw
    .withColumn("date_admission", F.to_date("date_admission"))
    .withColumn("duree_minutes", F.col("duree_minutes").cast("int")))
patients = patients_raw.withColumn("age", F.col("age").cast("int"))
admissions.printSchema()

## 2. Contrôles de qualité
Repérer : identifiants d’admission dupliqués, services vides, durées négatives, patients absents du référentiel. **Les compter avant de supprimer.**

In [ ]:
admissions.groupBy("admission_id").count().filter("count > 1").show()
admissions.filter((F.col("service") == "") | (F.col("duree_minutes") < 0)).show()
admissions.join(patients, "patient_id", "left_anti").show()

## 3. Table fiabilisée
Supprimer doublons exacts, puis exclure les données impossibles / incomplètes ; ne garder que les admissions ayant un patient connu. Pour un pipeline réel, conserver les rejets dans une table dédiée.

In [ ]:
admissions_clean = (admissions.dropDuplicates()
    .filter(F.col("service").isNotNull() & (F.col("service") != "") & (F.col("duree_minutes") >= 0))
    .join(patients.select("patient_id"), "patient_id", "inner"))
print("Brut :", admissions.count(), "| Nettoyé :", admissions_clean.count())

## 4. Jointure
Récupérez l’âge et le sexe de chaque patient. Comparez `inner` et `left` sur le petit jeu de données.

In [ ]:
visites_enrichies = admissions_clean.join(patients, "patient_id", "left")
visites_enrichies.select("admission_id", "service", "age", "duree_minutes").show(8)

## 5. Mini-défi
Quel service a le plus d’admissions ? Quelle est la durée moyenne par service ?

In [ ]:
visites_enrichies.groupBy("service").agg(F.count("*").alias("admissions"), F.round(F.avg("duree_minutes"), 1).alias("duree_moyenne_min")).orderBy(F.desc("admissions")).show()

### Discussion
Pourquoi vaut-il mieux compter et expliquer les rejets plutôt que supprimer silencieusement les lignes ? Pourquoi des données **réelles** de patients ne seraient-elles pas appropriées dans un dépôt GitHub public ?